# Know Your Bracket — what you'll actually face at *your* rating

Most meta write-ups for this competition show you the **top of the leaderboard**. That's useful if
you're already there. If you're sitting at 650 or 780, it's describing someone else's games.

The matchmaker pairs you with agents near **your own** rating, so the decks you meet at 650 are not
the decks the 1200s meet. This notebook answers the question you actually have:

> **I'm at rating X. What decks will I run into, and what beats them?**

It reads real public games straight off the ladder, works out which deck each side played, and
reports:

1. **Usage by bracket** — the most common decks in each 100-point band, from 500 up to 1100+.
2. **A "what beats what" grid** — matchup win rates, pooled across the ladder so the numbers have
   enough games behind them to mean something.
3. **A short read** — the few things the data says plainly.

Everything is recomputed from the raw games when you run it. Nothing here is pasted in.

**How to use it:** just run all cells. If you only care about your own bracket, scroll to the usage
table and find your band. To trade accuracy for speed, lower `TEAMS_PER_BAND` in the config cell.

## Setup

Two things this needs: the competition dataset attached (for `EN_Card_Data.csv`, to turn card IDs
into names), and **Internet on** (to read public games).

The episode and replay endpoints need a recent Kaggle SDK, which the default image doesn't always
carry — so we upgrade it first. **If the check below says to restart, restart the session and run
everything again.**

In [ ]:
get_ipython().run_line_magic("pip", 'install -q --upgrade "kaggle==2.2.3"')

In [ ]:
import importlib.metadata
import kaggle

print("Kaggle SDK:", importlib.metadata.version("kaggle"))
missing = [m for m in ("competition_team_submissions", "competition_list_episodes",
                       "competition_episode_replay") if not hasattr(kaggle.api, m)]
if missing:
    raise RuntimeError(
        f"This SDK is missing {missing}. Restart the session (Run > Restart session), "
        "then run all cells again — the upgrade above only takes effect after a restart."
    )
print("all required endpoints available")

In [ ]:
import glob
import json
import math
import os
import random
import re
import time
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

COMPETITION = "pokemon-tcg-ai-battle"

# --- knobs you might want to touch -------------------------------------------------
QUICK_TEST        = False  # False = the real run (~20 teams/bracket, ~25 min).
                           # Flip to True for a ~3 min smoke test that proves it all works.
TEAMS_PER_BAND    = 3 if QUICK_TEST else 20   # teams per bracket. Higher = tighter, slower.
REQUEST_INTERVAL  = 1.5    # seconds between API calls. Politeness; also avoids rate limits.
TIME_BUDGET_MIN   = 5 if QUICK_TEST else 35   # hard stop; the notebook can't run away with itself.
CLUSTER_THRESHOLD = 0.85   # how similar two decks must be to count as the same archetype.
MIN_GAMES_PER_CELL = 2 if QUICK_TEST else 10   # blank any matchup cell thinner than this.
                           # (kept low in QUICK_TEST only so the grid renders and you can see it
                           #  working — at that size the numbers mean nothing.)
MIN_DECKS_PER_BAND = 5     # brackets with fewer decks than this are reported but not ranked.
# ------------------------------------------------------------------------------------

BANDS = [
    (1100, 10_000, "1100+"),
    (1000, 1100, "1000-1099"),
    (900, 1000, "900-999"),
    (800, 900, "800-899"),
    (700, 800, "700-799"),
    (600, 700, "600-699"),
    (500, 600, "500-599"),
]
BAND_ORDER = [b[2] for b in BANDS]

def band_of(score):
    for lo, hi, name in BANDS:
        if lo <= score < hi:
            return name
    return None

plt.rcParams.update({"figure.dpi": 120, "font.size": 10, "axes.grid": True, "grid.alpha": 0.25,
                     "axes.spines.top": False, "axes.spines.right": False})
INK, ACCENT, MUTED = "#1b2a4a", "#c1440e", "#8a94a6"
print("config loaded")

### Card names

`EN_Card_Data.csv` maps card IDs to names. We also keep each Pokémon's HP, which is how we name a
deck later: the biggest Pokémon a deck reliably runs is, in practice, the thing it's built around.

In [ ]:
def find_card_csv():
    for p in ["/kaggle/input/pokemon-tcg-ai-battle/EN_Card_Data.csv",
              "/kaggle/input/competitions/pokemon-tcg-ai-battle/EN_Card_Data.csv"]:
        if os.path.exists(p):
            return p
    for pattern in ("/kaggle/input/**/EN_Card_Data.csv",   # anywhere in the attached inputs
                    "**/EN_Card_Data.csv"):                 # or alongside the notebook, if run locally
        hits = glob.glob(pattern, recursive=True)
        if hits:
            return hits[0]
    raise FileNotFoundError("EN_Card_Data.csv not found — attach the competition data to this notebook.")

_raw = pd.read_csv(find_card_csv(), encoding="utf-8-sig")

# Find the columns by what they CONTAIN, not by where we hope they sit. Guessing a fixed position
# for "Category" produced a table where nothing was recognised as a Pokemon, so every deck came out
# named "unknown". Sniffing is a few more lines and doesn't care how the file is laid out.
def _find_id_col(df):
    for n in ("Card ID", "card_id", "CardId", "id"):
        if n in df.columns:
            return df[n]
    for c in df.columns:                       # else: first integer-ish, all-unique column
        v = pd.to_numeric(df[c], errors="coerce")
        if v.notna().mean() > 0.95 and v.dropna().is_unique:
            return v
    return df.iloc[:, 0]

def _find_name_col(df, exclude):
    for n in ("Card Name", "card_name", "Name", "name"):
        if n in df.columns:
            return df[n]
    for c in df.columns:                       # else: the wordiest text column
        if c in exclude:
            continue
        s = df[c].astype(str)
        if s.str.contains(r"[A-Za-z]").mean() > 0.8 and s.str.len().mean() > 3:
            return s
    return df.iloc[:, 1]

def _find_hp_col(df):
    for n in ("HP", "hp", "Hp"):
        if n in df.columns:
            return pd.to_numeric(df[n], errors="coerce")
    best, best_score = None, 0                 # else: numeric column shaped like Pokemon HP
    for c in df.columns:
        v = pd.to_numeric(df[c], errors="coerce")
        plausible = v.between(20, 400) & (v % 10 == 0)
        score = plausible.mean()
        if score > best_score:
            best, best_score = v, score
    return best if best_score > 0.25 else pd.Series(0, index=df.index)

_id = _find_id_col(_raw)
cards = pd.DataFrame({
    "card_id": pd.to_numeric(_id, errors="coerce"),
    "name":    _find_name_col(_raw, exclude={getattr(_id, "name", None)}).astype(str),
    "hp":      _find_hp_col(_raw).fillna(0),
}).dropna(subset=["card_id"])
cards["card_id"] = cards["card_id"].astype(int)

# A Pokemon is the thing with HP. Do NOT test the stage/category text for "Pokemon": the stage
# column reads "Basic Pokemon"/"Stage 1 Pokemon" but ALSO "Pokemon Tool", which is a Trainer card —
# so that test is wrong in both directions. HP is unambiguous.
def _text(v):
    """Coerce a cell to a plain string. pandas 3 keeps missing values as NaN through
    .astype(str) (pandas 2 turned them into the string 'nan'), so anything that later calls a
    string method has to be defended here rather than trusting the dtype."""
    if v is None or (isinstance(v, float) and v != v):
        return ""
    return str(v).strip()

CARD_NAME = {cid: _text(nm) for cid, nm in zip(cards.card_id, cards.name)}
CARD_HP   = dict(zip(cards.card_id, cards.hp))
IS_POKEMON = {cid: bool(hp and hp > 0) for cid, hp in zip(cards.card_id, cards.hp)}

# Evolution chains, so a deck gets named after the card it evolves INTO rather than whichever
# middle stage happens to look distinctive ("Kadabra" when everyone calls the deck Alakazam).
_prev_col = next((c for c in _raw.columns if "previous" in str(c).lower()), None)
CHILDREN = defaultdict(set)
if _prev_col is not None:
    for child, parent in zip(cards.name, _raw[_prev_col].reindex(cards.index)):
        p, c = _text(parent), _text(child)
        if p and c and p.lower() not in ("n/a", "none"):
            CHILDREN[p].add(c)

# Whole evolution families. Disambiguating two same-named clusters has to exclude the entire
# line, not just the one card — otherwise the "second" signature card is simply the primary's
# own pre-evolution ("Mega Starmie ex + Staryu"), which distinguishes nothing.
_adj = defaultdict(set)
for _p, _cs in CHILDREN.items():
    for _c in _cs:
        _adj[_p].add(_c)
        _adj[_c].add(_p)
FAMILY = {}
for _start in list(_adj):
    if _start in FAMILY:
        continue
    _stack, _seen = [_start], set()
    while _stack:
        _x = _stack.pop()
        if _x in _seen:
            continue
        _seen.add(_x)
        _stack.extend(_adj[_x] - _seen)
    _fs = frozenset(_seen)
    for _x in _fs:
        FAMILY[_x] = _fs

n_pok = sum(IS_POKEMON.values())
print(f"{len(cards):,} cards · {n_pok:,} Pokemon · {len(CHILDREN):,} evolution links")
if n_pok < 50:
    raise RuntimeError(
        f"Only {n_pok} Pokemon detected — the card file isn't being read as expected, and every "
        f"deck would come out unnamed. Columns seen: {list(_raw.columns)[:9]}"
    )

## Reading real games off the ladder

For each sampled team we take one recent completed public game and read **both** decks out of the
game record. A replay stores each side's submitted 60-card deck directly, so these are exact lists —
not guesses reconstructed from the cards that happened to be revealed.

The sampled team's own deck is what we attribute to their bracket. Both decks feed the matchup grid.

Requests are spaced out deliberately. If the API rate-limits us, that team is skipped rather than
retried — a smaller honest sample beats hammering the service.

In [ ]:
import kaggle
from kaggle.api.kaggle_api_extended import ApiGetLeaderboardRequest

api = kaggle.api
_last_call = [0.0]
FAILED = object()   # distinct from None: several of these endpoints return None ON SUCCESS
API_ERRORS = Counter()

def paced(fn, *a, **kw):
    """Space out API calls, and swallow failures so one bad team can't kill a long run.

    Returns the FAILED sentinel on error — deliberately not None, because
    `competition_episode_replay` returns None when it *succeeds*, and conflating the two
    silently throws away every game you fetched.
    """
    wait = REQUEST_INTERVAL - (time.monotonic() - _last_call[0])
    if wait > 0:
        time.sleep(wait)
    _last_call[0] = time.monotonic()
    try:
        return fn(*a, **kw)
    except Exception as exc:
        API_ERRORS[f"{getattr(fn, '__name__', 'call')}: {type(exc).__name__}"] += 1
        return FAILED

def fetch_leaderboard():
    rows, token, seen = [], None, set()
    with api.build_kaggle_client() as client:
        while True:
            req = ApiGetLeaderboardRequest()
            req.competition_name = COMPETITION
            req.page_size = 200
            if token:
                req.page_token = token
            resp = paced(client.competitions.competition_api_client.get_leaderboard, req)
            if resp is FAILED or resp is None:
                break
            for s in (resp.submissions or []):
                d = s.to_dict() if hasattr(s, "to_dict") else dict(getattr(s, "__dict__", {}))
                # key spellings drift between SDK versions — accept any of them
                def pick(*names):
                    for n in names:
                        for key in (n, n.lstrip("_")):
                            if d.get(key) is not None:
                                return d[key]
                    return None
                tid = pick("teamId", "team_id", "TeamId")
                name = pick("teamName", "team_name", "TeamName") or ""
                score = pick("score", "publicScore", "public_score")
                try:
                    score = float(str(score).replace(",", ""))
                except (TypeError, ValueError):
                    continue
                if tid is not None:
                    rows.append({"team_id": int(tid), "team_name": str(name), "score": score})
            token = str(resp.next_page_token or "")
            if not token or token in seen:
                break
            seen.add(token)
    df = pd.DataFrame(rows).drop_duplicates("team_id")
    df["band"] = df["score"].map(band_of)
    return df.dropna(subset=["band"])

lb = fetch_leaderboard()
print(f"leaderboard: {len(lb):,} teams with a score of 500+")
print(lb.groupby("band").size().reindex(BAND_ORDER).fillna(0).astype(int).to_string())

In [ ]:
def exact_decks(replay):
    """Both sides' submitted 60-card decks, straight out of the game record."""
    steps = replay.get("steps") or []
    if len(steps) < 2 or not isinstance(steps[1], list):
        return None
    out = []
    for seat in range(2):
        try:
            action = steps[1][seat].get("action")
        except Exception:
            return None
        if not (isinstance(action, list) and len(action) == 60 and all(isinstance(x, int) for x in action)):
            return None
        out.append(action)
    return out

def newest_public_episode(submission_id):
    eps = paced(api.competition_list_episodes, int(submission_id))
    eps = [] if eps is FAILED or eps is None else eps
    ok = [e for e in eps
          if "PUBLIC" in str(getattr(e, "type", "")).upper()
          and "COMPLET" in str(getattr(e, "state", "")).upper()
          and getattr(e, "id", None) is not None]
    ok.sort(key=lambda e: int(e.id), reverse=True)
    return int(ok[0].id) if ok else None

def best_submission(team_id, score):
    subs = paced(api.competition_team_submissions, int(team_id))
    subs = [] if subs is FAILED or subs is None else subs
    cands = []
    for s in subs:
        sid = getattr(s, "id", None)
        if sid is None:
            continue
        ps = getattr(s, "public_score", None)
        try:
            ps = float(ps)
        except (TypeError, ValueError):
            ps = None
        cands.append((int(sid), ps))
    if not cands:
        return None
    # the submission whose public score is closest to the team's current standing
    return min(cands, key=lambda c: abs(c[1] - score) if c[1] is not None else 1e9)[0]

def sample_one_team(row, tmpdir="/kaggle/temp/kyb" if os.path.isdir("/kaggle/temp") else "/tmp/kyb"):
    """Returns (game_dict, None) on success or (None, reason) so skips can be explained."""
    sid = best_submission(row.team_id, row.score)
    if sid is None:
        return None, "no submission"
    eid = newest_public_episode(sid)
    if eid is None:
        return None, "no public completed episode"

    os.makedirs(tmpdir, exist_ok=True)
    for stale in glob.glob(os.path.join(tmpdir, "*.json")):
        os.remove(stale)
    # NOTE: this endpoint returns None on SUCCESS — the written file is the only success signal.
    paced(api.competition_episode_replay, eid, path=tmpdir, quiet=True)
    files = glob.glob(os.path.join(tmpdir, "*.json"))
    if not files:
        return None, "replay download failed"
    try:
        with open(files[0]) as fh:
            replay = json.load(fh)
    except Exception:
        return None, "replay unreadable"
    finally:
        for f in files:
            try:
                os.remove(f)
            except OSError:
                pass
    decks = exact_decks(replay)
    if decks is None:
        return None, "no 60-card decks in replay"
    info = replay.get("info") or {}
    names = [str(n) for n in (info.get("TeamNames") or [])]
    rewards = replay.get("rewards") or []
    if len(rewards) != 2 or rewards[0] is None or rewards[1] is None or rewards[0] == rewards[1]:
        winner = None                      # ties and crashed games carry no matchup signal
    else:
        winner = 0 if rewards[0] > rewards[1] else 1
    # which seat is the team we sampled? (falls back to seat 0 if names don't line up)
    mine = 0
    for i, n in enumerate(names[:2]):
        if n.strip().casefold() == str(row.team_name).strip().casefold():
            mine = i
    return {"episode_id": info.get("EpisodeId"), "band": row.band, "mine": mine,
            "decks": decks, "winner": winner}, None
print("readers defined")

In [ ]:
rng = random.Random(20260721)
picks = []
for band in BAND_ORDER:
    pool = lb[lb.band == band]
    if len(pool):
        picks += list(pool.sample(min(TEAMS_PER_BAND, len(pool)), random_state=7).itertuples())
rng.shuffle(picks)          # spread failures evenly across bands if we run out of time

print(f"sampling {len(picks)} teams across {len(BAND_ORDER)} brackets "
      f"(~{TIME_BUDGET_MIN} min budget)\n")

games, t0, skips = [], time.time(), Counter()
for i, row in enumerate(picks, 1):
    if (time.time() - t0) / 60 > TIME_BUDGET_MIN:
        print(f"\ntime budget reached — stopping at {i-1}/{len(picks)} teams")
        break
    g, why = sample_one_team(row)
    if g is None:
        skips[why] += 1
    else:
        games.append(g)
    if i % 10 == 0 or i == len(picks):
        print(f"  {i}/{len(picks)} teams · {len(games)} games read · {sum(skips.values())} skipped "
              f"· {(time.time()-t0)/60:.1f} min")

print(f"\n{len(games)} games read, {sum(skips.values())} skipped")
for why, n in skips.most_common():
    print(f"    {n:3d}  {why}")
for err, n in API_ERRORS.most_common(5):
    print(f"    [api] {n:3d}  {err}")
if not games:
    print("\nNothing came back. The skip reasons above say where it broke — that's the thing to fix.")

# One short line, so you can read the result out loud instead of copying anything.
worst = skips.most_common(1)[0][0] if skips else "none"
print("\n" + "=" * 58)
print(f"STATUS  read={len(games)}  skipped={sum(skips.values())}  worst-skip={worst}")
print("=" * 58)

## Naming the decks

Rather than maintain a hand-written list of "if it has card X it's deck Y" rules — which quietly goes
stale every time the field moves — we let the decks group themselves.

Two decks are treated as the same archetype when their card lists are close enough (cosine similarity
over card counts). Each resulting group is then named after the **highest-HP Pokémon that shows up in
most of its lists**, which is nearly always the card the deck is built around.

The upshot: a new deck that nobody has written a rule for still gets found and named.

In [ ]:
def cosine(a, b):
    ca, cb = Counter(a), Counter(b)
    dot = sum(v * cb[k] for k, v in ca.items())
    na = math.sqrt(sum(v * v for v in ca.values()))
    nb = math.sqrt(sum(v * v for v in cb.values()))
    return dot / (na * nb) if na and nb else 0.0

def cluster(decks, threshold):
    """Group similar decks (union-find over the similarity graph)."""
    parent = list(range(len(decks)))
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    for i in range(len(decks)):
        for j in range(i + 1, len(decks)):
            if find(i) != find(j) and cosine(decks[i], decks[j]) >= threshold:
                parent[max(find(i), find(j))] = min(find(i), find(j))
    return [find(i) for i in range(len(decks))]

def name_cluster(members, doc_freq, n_groups, exclude=()):
    """Name a group after the Pokemon most *distinctive* to it.

    Naming by "biggest Pokemon" sounds sensible and isn't: support Pokemon with huge HP show up
    in half the decks in the format, so the deck everyone calls Alakazam comes out named after
    whatever fat utility card it happens to run. Instead we score each Pokemon by how common it is
    *inside* this group against how many groups it appears in at all — the card that separates this
    deck from every other deck wins. Ubiquitous support cards score near zero automatically.
    """
    n = len(members)
    seen = Counter()
    copies = Counter()
    for deck in members:
        for cid in set(deck):
            if IS_POKEMON.get(cid):
                seen[cid] += 1
        for cid in deck:
            if IS_POKEMON.get(cid):
                copies[cid] += 1
    if not seen:   # no Pokemon recognised at all — name it after its most-run card instead of
        any_card = Counter()                     # dropping a useless "unknown" on the reader
        for deck in members:
            any_card.update(deck)
        return CARD_NAME.get(any_card.most_common(1)[0][0], "unnamed deck") if any_card else "unnamed deck"
    core = [cid for cid, c in seen.items() if c >= 0.6 * n] or list(seen)
    if exclude:   # for disambiguating two clusters that share a headline Pokemon
        core = [c for c in core if CARD_NAME.get(c) not in exclude] or core

    def distinctiveness(cid):
        inside = seen[cid] / n                                   # how reliably this group runs it
        spread = math.log(n_groups / (1 + doc_freq.get(cid, 0)))  # how rare it is elsewhere
        return inside * spread

    # Two signals, and the order matters. Distinctiveness alone picks whichever card is most unique
    # to the deck — often a basic or middle evolution ("Kadabra"), not what anyone calls the deck.
    # HP alone picks fat support Pokemon that half the format runs. So: use distinctiveness to find
    # the deck's signature *line*, then take the biggest card in it — the one it's named after.
    best = max(distinctiveness(c) for c in core)
    signature = [c for c in core if distinctiveness(c) >= 0.6 * best] or core

    # Rank by how many copies the deck runs, not by how big the Pokemon is. A deck plays 3-4 of the
    # line it's built on and exactly one splashable support — so copy count finds the engine, while
    # HP just finds the fattest card in the list and labels half the format after the same
    # generic support ex. (Tested both ways on real data: HP-first mislabels once samples get small,
    # copies-first holds up.)
    pick = max(signature, key=lambda c: (copies[c] / n, CARD_HP.get(c, 0)))
    name = CARD_NAME.get(pick, "unnamed deck")

    # Follow the evolution chain up to whatever this deck actually evolves into, so a group
    # doesn't end up labelled with a middle stage the deck merely passes through.
    present = {CARD_NAME.get(c) for c in seen}
    for _ in range(3):                      # chains are at most Basic -> Stage 1 -> Stage 2
        nxt = [c for c in CHILDREN.get(name, ()) if c in present and c not in exclude]
        if not nxt:
            break
        name = max(nxt, key=lambda c: max((CARD_HP.get(i, 0) for i in seen
                                           if CARD_NAME.get(i) == c), default=0))
    return name

flat = [d for g in games for d in g["decks"]]
labels = cluster(flat, CLUSTER_THRESHOLD) if flat else []
groups = defaultdict(list)
for lab, deck in zip(labels, flat):
    groups[lab].append(deck)
# how many distinct archetypes each Pokemon turns up in — the "is this card everywhere?" signal
doc_freq = Counter()
for ms in groups.values():
    for cid in {c for deck in ms for c in deck if IS_POKEMON.get(c)}:
        doc_freq[cid] += 1
naming = {lab: name_cluster(ms, doc_freq, max(len(groups), 2)) for lab, ms in groups.items()}
# two groups can land on the same headline Pokemon; keep them distinct but readable
# Two clusters can share a headline Pokemon while being genuinely different decks — a Crustle
# that races and a Crustle that mills are not the same thing to a reader. Calling them "build 1"
# and "build 2" hides exactly the distinction that matters, so name the collision by each
# cluster's SECOND signature card instead.
dupes = Counter(naming.values())
for lab in sorted(groups, key=lambda x: -len(groups[x])):
    if dupes[naming[lab]] > 1:
        second = name_cluster(groups[lab], doc_freq, max(len(groups), 2),
                              exclude=FAMILY.get(naming[lab], {naming[lab]}))
        if second and second != naming[lab]:
            naming[lab] = f"{naming[lab]} + {second}"

k = 0
for g in games:
    g["labels"] = [naming[labels[k]], naming[labels[k + 1]]]
    k += 2

print(f"{len(flat)} decks -> {len(groups)} archetypes")

## 1. What you'll face, bracket by bracket

One deck per sampled team, counted in that team's bracket. **Find your rating and read across.**

In [ ]:
rows = []
for g in games:
    rows.append({"band": g["band"], "deck": g["labels"][g["mine"]]})
per_band = pd.DataFrame(rows)

if len(per_band):
    tbl = (per_band.groupby(["band", "deck"]).size().rename("teams").reset_index())
    tot = per_band.groupby("band").size().rename("band_total")
    tbl = tbl.join(tot, on="band")
    tbl["share"] = (100 * tbl.teams / tbl.band_total).round(1)
    tbl = tbl.sort_values(["band", "teams"], ascending=[True, False])

    for band in BAND_ORDER:
        sub = tbl[tbl.band == band]
        if not len(sub):
            continue
        n = int(sub.band_total.iloc[0])
        flag = "" if n >= MIN_DECKS_PER_BAND else "   (thin sample — read as a hint, not a fact)"
        print(f"\n=== {band}  ·  {n} decks{flag}")
        for r in sub.head(6).itertuples():
            print(f"    {r.share:5.1f}%  {int(r.teams):2d}x  {r.deck}")
else:
    print("no games were read — try re-running, or raise TEAMS_PER_BAND")

In [ ]:
if len(per_band):
    top = per_band.deck.value_counts().head(8).index.tolist()
    mat = pd.DataFrame(0.0, index=top, columns=[b for b in BAND_ORDER if (per_band.band == b).any()])
    for band in mat.columns:
        sub = per_band[per_band.band == band]
        if len(sub):
            share = sub.deck.value_counts(normalize=True) * 100
            for d in top:
                mat.loc[d, band] = float(share.get(d, 0.0))

    fig, ax = plt.subplots(figsize=(1.15 * len(mat.columns) + 4, 0.5 * len(top) + 2.2))
    im = ax.imshow(mat.values, cmap="YlGnBu", aspect="auto")
    ax.set_xticks(range(len(mat.columns)), mat.columns, rotation=30, ha="right")
    ax.set_yticks(range(len(top)), [t[:26] for t in top])
    for i in range(len(top)):
        for j in range(len(mat.columns)):
            v = mat.values[i, j]
            if v > 0:
                ax.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=8,
                        color="white" if v > mat.values.max() * 0.55 else INK)
    ax.set_title("Share of each bracket (%)", color=INK, loc="left", fontweight="bold")
    plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
    plt.tight_layout(); plt.show()

## 2. What beats what

Matchups are **pooled across every bracket**. That's deliberate: split seven ways, almost no pairing
would have enough games to say anything, and a win rate off four games is noise wearing a percentage
sign. Pooled, the common pairings have real weight behind them.

Read it as: **row deck's win rate against column deck.** Cells with fewer than
`MIN_GAMES_PER_CELL` games are left blank on purpose.

In [ ]:
pairs = defaultdict(lambda: [0, 0])   # (deck_a, deck_b) -> [wins_for_a, games]
for g in games:
    if g["winner"] is None:
        continue
    a, b = g["labels"]
    if a == b:
        continue
    wa = 1 if g["winner"] == 0 else 0
    pairs[(a, b)][0] += wa;  pairs[(a, b)][1] += 1
    pairs[(b, a)][0] += 1 - wa;  pairs[(b, a)][1] += 1

played = Counter()
for g in games:
    for d in g["labels"]:
        played[d] += 1
grid_decks = [d for d, _ in played.most_common(8)]

M = np.full((len(grid_decks), len(grid_decks)), np.nan)
N = np.zeros_like(M)
for i, a in enumerate(grid_decks):
    for j, b in enumerate(grid_decks):
        if i == j:
            continue
        w, n = pairs.get((a, b), [0, 0])
        N[i, j] = n
        if n >= MIN_GAMES_PER_CELL:
            M[i, j] = 100.0 * w / n

if np.isfinite(M).any():
    fig, ax = plt.subplots(figsize=(0.95 * len(grid_decks) + 4, 0.75 * len(grid_decks) + 3))
    im = ax.imshow(np.ma.masked_invalid(M), cmap="RdBu_r", vmin=25, vmax=75, aspect="auto")
    ax.set_xticks(range(len(grid_decks)), [d[:16] for d in grid_decks], rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(len(grid_decks)), [d[:22] for d in grid_decks], fontsize=8)
    for i in range(len(grid_decks)):
        for j in range(len(grid_decks)):
            if np.isfinite(M[i, j]):
                ax.text(j, i, f"{M[i,j]:.0f}", ha="center", va="center", fontsize=8,
                        color="white" if abs(M[i, j] - 50) > 18 else INK)
            elif i != j:
                ax.text(j, i, "·", ha="center", va="center", color=MUTED)
    ax.set_title("Row deck's win rate vs column deck (%)", color=INK, loc="left", fontweight="bold")
    plt.colorbar(im, ax=ax, fraction=0.035, label="row win %")
    plt.tight_layout(); plt.show()
    print(f"blank = fewer than {MIN_GAMES_PER_CELL} games between those two decks in this sample")
else:
    print(f"no matchup reached {MIN_GAMES_PER_CELL} games — raise TEAMS_PER_BAND for a fuller grid")

## 3. What the data says plainly

Generated from the numbers above — no editorialising.

In [ ]:
notes = []
if len(per_band):
    for band in BAND_ORDER:
        sub = per_band[per_band.band == band]
        if len(sub) >= MIN_DECKS_PER_BAND:
            vc = sub.deck.value_counts(normalize=True) * 100
            notes.append(f"At **{band}**, the deck you'll meet most is **{vc.index[0]}** "
                         f"({vc.iloc[0]:.0f}% of teams sampled there).")
    spread = {}
    for band in BAND_ORDER:
        sub = per_band[per_band.band == band]
        if len(sub) >= MIN_DECKS_PER_BAND:
            spread[band] = sub.deck.nunique() / len(sub)
    if len(spread) >= 2:
        widest = max(spread, key=spread.get); tightest = min(spread, key=spread.get)
        notes.append(f"**{widest}** is the most varied bracket and **{tightest}** the most "
                     f"concentrated — expect a narrower range of decks as you climb into "
                     f"{tightest}.")
best = [(a, b, 100.0 * w / n, n) for (a, b), (w, n) in pairs.items() if n >= MIN_GAMES_PER_CELL]
if best:
    a, b, p, n = max(best, key=lambda x: x[2])
    notes.append(f"The most lopsided matchup with enough games behind it: **{a}** beats "
                 f"**{b}** {p:.0f}% of the time (n={n}).")
if not notes:
    notes = ["Not enough games in this run to say anything solid. Raise `TEAMS_PER_BAND` and re-run."]

from IPython.display import Markdown, display
display(Markdown("\n\n".join(f"- {n}" for n in notes)))

## What this can and can't tell you

Worth being straight about, because a meta table that hides its limits is worse than no table:

- **It's a sample, not a census.** We read a bounded number of teams per bracket, so shares are
  estimates. Small brackets are flagged in the tables above.
- **One deck per team.** A team that's been climbing all week is one row, same as a team that
  submitted an hour ago.
- **Brackets come from current standing**, but the game we read may have been played a little
  earlier, so a few decks sit near a boundary.
- **Matchups are pooled across brackets**, so a matchup number describes the ladder overall, not
  specifically your bracket. Splitting it further would produce numbers too thin to trust.
- **Win rates reflect the pilots who actually played those decks.** A deck's record here is the deck
  *and* the agent flying it, together — not the deck's ceiling in better hands.
- **Deck names are inferred, not official.** Groups are named after the card that most distinguishes
  them, so a deck occasionally comes out under a different member of its evolution line than the
  name people use in conversation. The grouping is what matters; the label is a handle.
- **The meta moves.** This is a snapshot from whenever you ran it. Re-run it for a fresh one.

Anything unclear or wrong, say so in the comments — I'd rather fix it than have it quietly mislead
someone.